# Notebook 02 – Baseline Models

Train and evaluate four baselines:
1. Global Mean
2. User Mean
3. Item Mean
4. Bias Model (μ + b_u + b_i, SGD)


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.baselines  import GlobalMean, UserMean, ItemMean, BiasModel
from src.evaluation import rating_metrics

sns.set_style('whitegrid')
os.makedirs('../results', exist_ok=True)
SEED = 42
np.random.seed(SEED)

In [ ]:
train = pd.read_parquet('../data/train.parquet')
test  = pd.read_parquet('../data/test.parquet')

n_users  = int(train['user_idx'].max()) + 1
n_movies = int(train['movie_idx'].max()) + 1
print(f'Train: {len(train):,}   Test: {len(test):,}')
print(f'n_users={n_users:,}  n_movies={n_movies:,}')

## 2.1 Fit & evaluate all baselines

In [ ]:
results = {}

results['GlobalMean'] = GlobalMean().fit(train).evaluate(test)
results['UserMean']   = UserMean().fit(train).evaluate(test)
results['ItemMean']   = ItemMean().fit(train).evaluate(test)

print('\nTraining BiasModel ...')
bm = BiasModel(n_users=n_users, n_items=n_movies, lam=0.1, lr=0.005, n_epochs=20)
bm.fit(train)
results['BiasModel'] = bm.evaluate(test)

df_results = pd.DataFrame(results).T
print('\n--- Baseline Results ---')
print(df_results.to_string())

## 2.2 RMSE comparison plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_results['RMSE'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Baseline Models – Test RMSE')
axes[0].set_ylabel('RMSE (lower is better)')
axes[0].set_xticklabels(df_results.index, rotation=30, ha='right')
for i, v in enumerate(df_results['RMSE']):
    axes[0].text(i, v + 0.003, f'{v:.4f}', ha='center', fontsize=9)

df_results['MAE'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Baseline Models – Test MAE')
axes[1].set_ylabel('MAE (lower is better)')
axes[1].set_xticklabels(df_results.index, rotation=30, ha='right')
for i, v in enumerate(df_results['MAE']):
    axes[1].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/baseline_rmse.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.3 Bias model – learned bias distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(bm.b_u, bins=50, color='coral', edgecolor='white')
axes[0].set_title('User Biases b_u')
axes[0].set_xlabel('Bias value')
axes[0].set_ylabel('Count')

axes[1].hist(bm.b_i, bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Item Biases b_i')
axes[1].set_xlabel('Bias value')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../results/bias_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_results.to_csv('../results/baseline_results.csv')
print('Saved to results/baseline_results.csv')